In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
from IPython.display import display

# Complete multi-sector ticker universe (53 symbols)
# Replacements applied:
# - BMWYY -> BMW.DE (Frankfurt XETRA)
# - K (Kellanova privatized) -> GIS (General Mills)
# - SKX (quote 404) -> DECK (Deckers Outdoor)
SUPER_BOWL_TICKERS = [
    # Market & Sector Controls
    "^GSPC",
    "XLP",
    "XLY",
    "XLC",
    # Beverages & CPG Food
    "BUD",
    "PEP",
    "KO",
    "KDP",
    "KHC",
    "GIS",
    "HSY",
    "MDLZ",
    "HRL",
    "PG",
    "UL",
    # Automotive
    "TM",
    "GM",
    "F",
    "STLA",
    "HMC",
    "005380.KS",
    "000270.KS",
    "VWAGY",
    "BMW.DE",
    "MBGYY",
    # Tech, Streaming & Media
    "AMZN",
    "GOOGL",
    "MSFT",
    "CRM",
    "WDAY",
    "DIS",
    "NFLX",
    "CMCSA",
    "PARA",
    "FOXA",
    # Telecom, Travel & Delivery
    "VZ",
    "TMUS",
    "T",
    "DASH",
    "UBER",
    "BKNG",
    "EXPE",
    # Financial Services & Fintech
    "MS",
    "INTU",
    "HRB",
    "RKT",
    "COIN",
    # QSR, Apparel & Logistics
    "MCD",
    "YUM",
    "WEN",
    "NKE",
    "DECK",
    "FDX",
]

START_DATE = "2000-01-01"
END_DATE = "2026-03-01"

print(
    f"Configured batch ingestion for {len(SUPER_BOWL_TICKERS)} tickers ({START_DATE} to {END_DATE})."
)

Configured batch ingestion for 53 tickers (2000-01-01 to 2026-03-01).


In [10]:
print("Executing batch download via yfinance...")
raw_data = yf.download(
    tickers=SUPER_BOWL_TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    group_by="column",
)

# Extract Adjusted Close prices (fallback to Close if needed)
if "Adj Close" in raw_data.columns:
    prices_df = raw_data["Adj Close"].copy()
else:
    prices_df = raw_data["Close"].copy()

# Drop any symbols that returned completely empty series
prices_df = prices_df.dropna(how="all", axis=1)
print(
    f"Successfully retrieved price series for {prices_df.shape[1]} of {len(SUPER_BOWL_TICKERS)} symbols."
)

# Compute daily logarithmic returns: r_t = ln(P_t / P_{t-1})
daily_log_returns = np.log(prices_df / prices_df.shift(1))

# Define output filenames
csv_path = "superbowl_advertisers_daily_returns_2000_2026.csv"
parquet_path = "superbowl_advertisers_daily_returns_2000_2026.parquet"

# Export universal CSV
daily_log_returns.to_csv(csv_path)
print(f"Exported daily returns to CSV: {csv_path}")

# Export high-performance Parquet
try:
    daily_log_returns.to_parquet(parquet_path, engine="pyarrow")
    print(f"Exported daily returns to Parquet: {parquet_path}")
except Exception as err:
    print(f"Parquet engine skipped ({err}). CSV backup available.")

Executing batch download via yfinance...


[*********************100%***********************]  53 of 53 completed


Successfully retrieved price series for 53 of 53 symbols.
Exported daily returns to CSV: superbowl_advertisers_daily_returns_2000_2026.csv
Exported daily returns to Parquet: superbowl_advertisers_daily_returns_2000_2026.parquet


In [11]:
from IPython.display import display

# Reload and inspect the exported dataset
df_verified = pd.read_parquet(parquet_path, engine="pyarrow")

print("=== Dataset Diagnostics ===")
print(f"Trading Days (Rows): {df_verified.shape[0]:,}")
print(f"Tickers (Columns):    {df_verified.shape[1]}")
print(
    f"Date Range:           {df_verified.index.min().strftime('%Y-%m-%d')} to {df_verified.index.max().strftime('%Y-%m-%d')}"
)

print("\n--- Recent Daily Returns Preview ---")
display(
    df_verified[
        ["^GSPC", "XLP", "BUD", "PEP", "KO", "TM", "BMW.DE", "GIS", "DECK"]
    ].tail(5)
)

# Check integrity of replacement tickers
replacements = ["BMW.DE", "GIS", "DECK"]
for sym in replacements:
    status = "Retrieved" if sym in df_verified.columns else "Missing"
    print(f"  {sym}: {status}")

=== Dataset Diagnostics ===
Trading Days (Rows): 6,796
Tickers (Columns):    53
Date Range:           2000-01-03 to 2026-02-27

--- Recent Daily Returns Preview ---


Ticker,^GSPC,XLP,BUD,PEP,KO,TM,BMW.DE,GIS,DECK
Date,,,,,,,,,
2026-02-23,-0.010440,0.012213,0.018673,0.020285,0.008978,-0.006574,-0.029437,0.016889,-0.004391
2026-02-24,0.007622,0.008617,0.008216,0.007222,0.001984,0.002559,0.008855,0.000881,0.023170
2026-02-25,0.008103,-0.008168,-0.025998,-0.002185,-0.003102,0.012865,-0.004077,-0.019565,-0.012648
2026-02-26,-0.005380,-0.001687,0.011261,-0.009443,0.000373,0.003675,0.008137,0.008718,-0.000084
2026-02-27,-0.004349,0.012859,0.018326,0.012807,0.013082,-0.000990,0.006955,0.006655,-0.018084


  BMW.DE: Retrieved
  GIS: Retrieved
  DECK: Retrieved
